# Verify Signals — Indicator & Crossover Audit

Proves the backtest engine does what we think:
1. NT’s internal indicators match independent pandas calculations
2. Fills happen exactly when crossovers occur — no missed signals, no phantom trades

In [ ]:
# ── Cell 1: Imports + config ───────────────────────────────────────────
#
# ★ Change STRATEGY, periods, and instrument to verify different setups. ★

from decimal import Decimal

import numpy as np
import pandas as pd

from nautilus_trader.indicators import ExponentialMovingAverage, SimpleMovingAverage
from nautilus_trader.model.data import BarType
from nautilus_trader.model.identifiers import InstrumentId, Venue
from nautilus_trader.persistence.catalog import ParquetDataCatalog

from src.backtesting import make_engine
from src.core import bar_type_str, with_venue_config
from src.strategies.ema_cross import EMACross, EMACrossConfig
from src.strategies.sma_cross import SMACross, SMACrossConfig

# ── What to verify ────────────────────────────────────────────────────
EXCHANGE         = "BINANCE"
INSTRUMENT_ID    = f"BTCUSDT-PERP.{EXCHANGE}"
BAR_INTERVAL     = "1d"
STARTING_CAPITAL = 10_000
TRADE_NOTIONAL   = Decimal("1000")
LEVERAGE         = 20

STRATEGY    = "SMACross"
FAST_PERIOD = 10
SLOW_PERIOD = 50

# Standard
BAR_TYPE_STR = bar_type_str(INSTRUMENT_ID, BAR_INTERVAL)
VENUE        = Venue(EXCHANGE)

# Files
CATALOG_PATH = "../data/catalog"

# Strategy factory — adds the configured strategy to the engine
def strategy_factory(eng):
    match STRATEGY:
        case "SMACross":
            cfg = SMACrossConfig(
                instrument_id=InstrumentId.from_str(INSTRUMENT_ID),
                bar_type=BarType.from_str(BAR_TYPE_STR),
                trade_notional=TRADE_NOTIONAL,
                fast_sma_period=FAST_PERIOD,
                slow_sma_period=SLOW_PERIOD,
            )
            eng.add_strategy(SMACross(cfg))
        case "EMACross":
            cfg = EMACrossConfig(
                instrument_id=InstrumentId.from_str(INSTRUMENT_ID),
                bar_type=BarType.from_str(BAR_TYPE_STR),
                trade_notional=TRADE_NOTIONAL,
                fast_ema_period=FAST_PERIOD,
                slow_ema_period=SLOW_PERIOD,
            )
            eng.add_strategy(EMACross(cfg))

MA_TYPE = "SMA" if STRATEGY == "SMACross" else "EMA"

print(f"Verifying: {STRATEGY} on {INSTRUMENT_ID} {BAR_INTERVAL}")
print(f"Periods: fast={FAST_PERIOD}, slow={SLOW_PERIOD}")
print("Imports OK")

In [ ]:
# ── Cell 2: Load data ────────────────────────────────────────────────
catalog    = ParquetDataCatalog(CATALOG_PATH)
instrument = catalog.instruments(instrument_ids=[INSTRUMENT_ID])[0]
bars       = catalog.bars(bar_types=[BAR_TYPE_STR])
instrument = with_venue_config(instrument, LEVERAGE)
CURRENCY   = instrument.settlement_currency

print(f"Instrument : {instrument.id}")
print(f"Bars       : {len(bars):,}")
print(f"Range      : {pd.Timestamp(bars[0].ts_event, unit='ns', tz='UTC'):%Y-%m-%d}")
print(f"           → {pd.Timestamp(bars[-1].ts_event, unit='ns', tz='UTC'):%Y-%m-%d}")

---
## Part 1: Indicator Comparison

Compute indicators two ways and verify they match bar-by-bar:
- **Manual:** pandas `rolling().mean()` (SMA) or `.ewm()` (EMA) on close prices
- **NT:** NautilusTrader's `SimpleMovingAverage` / `ExponentialMovingAverage` fed bar-by-bar via `.handle_bar()`

In [ ]:
# ── Cell 3: Compute indicators — manual (pandas) vs NT ───────────

# ── Manual calculation using pandas ──
closes = pd.Series([float(bar.close) for bar in bars])
timestamps = [pd.Timestamp(bar.ts_event, unit="ns", tz="UTC") for bar in bars]

if MA_TYPE == "SMA":
    manual_fast = closes.rolling(FAST_PERIOD).mean()
    manual_slow = closes.rolling(SLOW_PERIOD).mean()
else:  # EMA
    manual_fast = closes.ewm(span=FAST_PERIOD, adjust=False).mean()
    manual_slow = closes.ewm(span=SLOW_PERIOD, adjust=False).mean()

# ── NT indicator calculation (bar-by-bar, same approach as charts.py) ──
NT_MA = SimpleMovingAverage if MA_TYPE == "SMA" else ExponentialMovingAverage
nt_fast_ind = NT_MA(FAST_PERIOD)
nt_slow_ind = NT_MA(SLOW_PERIOD)

nt_fast_vals = []
nt_slow_vals = []
for bar in bars:
    nt_fast_ind.handle_bar(bar)
    nt_slow_ind.handle_bar(bar)
    nt_fast_vals.append(nt_fast_ind.value if nt_fast_ind.initialized else np.nan)
    nt_slow_vals.append(nt_slow_ind.value if nt_slow_ind.initialized else np.nan)

nt_fast = pd.Series(nt_fast_vals)
nt_slow = pd.Series(nt_slow_vals)

# ── Build comparison DataFrame ──
df = pd.DataFrame({
    "ts": timestamps,
    "close": closes.values,
    "open":  [float(b.open)  for b in bars],
    "high":  [float(b.high)  for b in bars],
    "low":   [float(b.low)   for b in bars],
    "manual_fast": manual_fast.values,
    "manual_slow": manual_slow.values,
    "nt_fast": nt_fast.values,
    "nt_slow": nt_slow.values,
})
df["single_price"] = (
    (df["open"] == df["high"]) &
    (df["high"] == df["low"]) &
    (df["low"]  == df["close"])
)

# Compare where both are non-NaN (separate masks — slow MA has longer warm-up)
fast_mask = df["manual_fast"].notna() & df["nt_fast"].notna()
slow_mask = df["manual_slow"].notna() & df["nt_slow"].notna()

fast_diff = (df.loc[fast_mask, "manual_fast"] - df.loc[fast_mask, "nt_fast"]).abs()
slow_diff = (df.loc[slow_mask, "manual_slow"] - df.loc[slow_mask, "nt_slow"]).abs()

atol = 1e-10 if MA_TYPE == "SMA" else 1e-8

print(f"{'\u2500' * 50}")
print(f"Indicator Comparison: {MA_TYPE} (manual pandas vs NT)")
print(f"{'\u2500' * 50}")
print(f"  Fast {MA_TYPE}({FAST_PERIOD}):  ({fast_mask.sum():,} bars compared)")
print(f"    Max diff    : {fast_diff.max():.2e}")
print(f"    Mean diff   : {fast_diff.mean():.2e}")
print(f"    Diffs > tol : {(fast_diff > atol).sum()}")
print(f"  Slow {MA_TYPE}({SLOW_PERIOD}):  ({slow_mask.sum():,} bars compared)")
print(f"    Max diff    : {slow_diff.max():.2e}")
print(f"    Mean diff   : {slow_diff.mean():.2e}")
print(f"    Diffs > tol : {(slow_diff > atol).sum()}")

fast_match = np.allclose(df.loc[fast_mask, "manual_fast"], df.loc[fast_mask, "nt_fast"], atol=atol)
slow_match = np.allclose(df.loc[slow_mask, "manual_slow"], df.loc[slow_mask, "nt_slow"], atol=atol)
INDICATORS_MATCH = fast_match and slow_match

if INDICATORS_MATCH:
    print(f"\n\u2705 PASS \u2014 indicators match (atol={atol})")
else:
    print(f"\n\u274c FAIL \u2014 indicators diverge beyond tolerance")
    if not fast_match:
        print(f"    Fast {MA_TYPE} mismatch")
    if not slow_match:
        print(f"    Slow {MA_TYPE} mismatch")

---
## Part 2: Crossover Detection

Identify every bar where the signal state changes, matching the strategy's logic:
- `fast >= slow` → BUY state
- `fast < slow` → SELL state
- A crossover = the bar where the state changes from the previous bar

In [ ]:
# ── Cell 4: Identify crossover points ─────────────────────────
#
# Signal state matches strategy logic:
#   fast >= slow → BUY state  (signal = 1)
#   fast <  slow → SELL state (signal = -1)

valid = df["nt_fast"].notna() & df["nt_slow"].notna()
df["signal"] = np.nan
df.loc[valid, "signal"] = np.where(
    df.loc[valid, "nt_fast"] >= df.loc[valid, "nt_slow"], 1, -1
)

# Detect state transitions (includes first valid bar, whose prev is NaN)
df["prev_signal"] = df["signal"].shift(1)
df["signal_change"] = (
    valid &
    (df["signal"] != df["prev_signal"]) &
    df["signal"].notna()
)

# Build crossovers table (exclude single-price bars)
cross_mask = df["signal_change"] & ~df["single_price"]
crossovers = df.loc[cross_mask, ["ts", "signal", "nt_fast", "nt_slow"]].copy()
crossovers["direction"] = crossovers["signal"].map({1.0: "BUY", -1.0: "SELL"})
crossovers["bar_idx"] = crossovers.index
crossovers = crossovers.reset_index(drop=True)

# Count crossovers blocked by single-price bars
blocked = (df["signal_change"] & df["single_price"]).sum()

print(f"Crossover points detected: {len(crossovers)}")
print(f"  Bullish (BUY):  {(crossovers['direction'] == 'BUY').sum()}")
print(f"  Bearish (SELL): {(crossovers['direction'] == 'SELL').sum()}")
if blocked:
    print(f"  Single-price bars skipped: {blocked}")
print()
print("First 5 crossovers:")
print(crossovers[["ts", "direction", "nt_fast", "nt_slow"]].head().to_string(index=False))
print()
print("Last 5 crossovers:")
print(crossovers[["ts", "direction", "nt_fast", "nt_slow"]].tail().to_string(index=False))

---
## Part 3: Backtest Fill Comparison

Run a backtest on the same data, extract fills, and compare against the crossovers above.

Expected fill pattern:
- **First crossover:** 1 entry fill
- **Subsequent crossovers:** 2 fills (close existing + new entry)
- **Strategy stop:** 1 close fill at end of backtest (if positioned)

In [ ]:
# ── Cell 5: Run backtest + extract fills ────────────────────
eng = make_engine(VENUE, instrument, bars, STARTING_CAPITAL)
strategy_factory(eng)
eng.run()

# Calculate statistics
account = eng.cache.account_for_venue(VENUE)
positions = eng.cache.position_snapshots() + eng.cache.positions()
analyzer = eng.portfolio.analyzer
analyzer.calculate_statistics(account, positions)

fills_report = eng.trader.generate_order_fills_report()
print(f"Fills: {len(fills_report)}")
print(f"Positions: {len(positions)}")
print(f"Total PnL: {analyzer.total_pnl(CURRENCY)}")

# Parse fill timestamps and sides
def _first_col(fr, candidates):
    return next((c for c in candidates if c in fr.columns), None)

if not fills_report.empty:
    ts_col   = _first_col(fills_report, ["ts_last", "ts_event", "ts_filled"])
    side_col = _first_col(fills_report, ["side", "order_side"])

    fills = fills_report.copy()
    fills["_ts"] = pd.to_datetime(fills[ts_col].astype("int64"), unit="ns", utc=True)
    fills["_side"] = fills[side_col].astype(str)

    print(f"\nFill timestamp column: {ts_col}")
    print(f"\nFirst 5 fills:")
    print(fills[["_ts", "_side"]].head().to_string())
    print(f"\nLast 5 fills:")
    print(fills[["_ts", "_side"]].tail().to_string())
else:
    fills = pd.DataFrame()
    print("No fills \u2014 strategy produced no signals")

eng.dispose()

In [ ]:
# ── Cell 6: Compare fills vs crossovers ─────────────────────

if fills.empty:
    print("No fills to compare.")
    FILLS_MATCH = False
else:
    # Map fill timestamps to bar indices.
    # In backtesting, fills happen when on_bar fires. The fill timestamp
    # may match either ts_event or ts_init of the bar.
    ts_event_map = {int(bar.ts_event): i for i, bar in enumerate(bars)}
    ts_init_map  = {int(bar.ts_init):  i for i, bar in enumerate(bars)}

    fill_bar_indices = []
    for _, row in fills.iterrows():
        fts = int(row[ts_col].value)  # Timestamp → nanoseconds
        if fts in ts_event_map:
            fill_bar_indices.append(ts_event_map[fts])
        elif fts in ts_init_map:
            fill_bar_indices.append(ts_init_map[fts])
        else:
            fill_bar_indices.append(None)
    fills["_bar_idx"] = fill_bar_indices

    # Mapping stats
    n_mapped = sum(1 for f in fill_bar_indices if f is not None)
    n_unmapped = sum(1 for f in fill_bar_indices if f is None)
    print(f"Fill-to-bar mapping: {n_mapped} mapped, {n_unmapped} unmapped")

    # Crossover bar indices (from Part 2)
    cross_bar_set = {idx + 1 for idx in crossovers["bar_idx"] if idx + 1 < len(bars)}

    # Group fills by bar
    mapped_fills = fills[fills["_bar_idx"].notna()].copy()
    mapped_fills["_bar_idx"] = mapped_fills["_bar_idx"].astype(int)
    fill_by_bar = mapped_fills.groupby("_bar_idx")
    fill_bar_set = set(mapped_fills["_bar_idx"])

    # ── Crossovers → fills check ──
    cross_with_fill = cross_bar_set & fill_bar_set
    cross_without_fill = cross_bar_set - fill_bar_set

    # ── Fills → crossovers check ──
    fills_at_cross = fill_bar_set & cross_bar_set
    fills_not_at_cross = fill_bar_set - cross_bar_set

    # Classify non-crossover fills
    extra_detail = []
    for bar_idx in sorted(fills_not_at_cross):
        group = fill_by_bar.get_group(bar_idx)
        sides = group["_side"].tolist()
        ts = df.loc[bar_idx, "ts"] if bar_idx < len(df) else "?"
        is_last = bar_idx == len(bars) - 1
        label = "(last bar — strategy stop)" if is_last else "(unexpected)"
        extra_detail.append(f"    bar {bar_idx} {ts}  fills={sides}  {label}")

    # ── Direction check ──
    direction_mismatches = []
    for _, crow in crossovers.iterrows():
        bidx = int(crow["bar_idx"]) + 1  # fills land one bar after crossover
        expected = crow["direction"]  # "BUY" or "SELL"
        if bidx not in fill_bar_set:
            continue
        group = fill_by_bar.get_group(bidx)
        fill_sides = group["_side"].tolist()
        # All fills at a crossover bar should match the crossover direction
        for side in fill_sides:
            if expected.upper() not in side.upper():
                direction_mismatches.append(
                    f"    bar {bidx} {crow['ts']}  expected={expected}  got={side}"
                )

    # ── Report ──
    print(f"\n{'\u2500' * 60}")
    print("Fill vs Crossover Comparison")
    print(f"{'\u2500' * 60}")
    print(f"  Total crossovers detected:       {len(cross_bar_set)}")
    print(f"  Total fill events:               {len(fills)}")
    print(f"  Unique bars with fills:          {len(fill_bar_set)}")
    print(f"  Crossovers with fills:           {len(cross_with_fill)}")
    print(f"  Crossovers WITHOUT fills:        {len(cross_without_fill)}")
    print(f"  Fill bars NOT at crossover:      {len(fills_not_at_cross)}")
    print(f"  Direction mismatches:            {len(direction_mismatches)}")

    if cross_without_fill:
        print(f"\n  \u26a0 Crossovers missing fills:")
        for bidx in sorted(cross_without_fill):
            row = df.loc[bidx]
            sig = "BUY" if row['signal'] == 1 else "SELL"
            print(f"    bar {bidx} {row['ts']}  signal={sig}")

    if extra_detail:
        print(f"\n  Fills at non-crossover bars:")
        for line in extra_detail:
            print(line)

    if direction_mismatches:
        print(f"\n  \u26a0 Direction mismatches:")
        for line in direction_mismatches:
            print(line)

    # Expected: all crossovers have fills, 0-1 extra (stop close),
    # 0 direction mismatches
    all_stop_fills = all(
        int(b) == len(bars) - 1 for b in fills_not_at_cross
    ) if fills_not_at_cross else True
    FILLS_MATCH = (
        len(cross_without_fill) == 0
        and len(direction_mismatches) == 0
        and all_stop_fills
        and n_unmapped == 0
    )

    if FILLS_MATCH:
        print(f"\n\u2705 PASS \u2014 all crossovers have fills, directions match")
    else:
        print(f"\n\u274c FAIL \u2014 discrepancies found (see details above)")

---
## Summary

In [ ]:
# ── Cell 7: Verdict ─────────────────────────────────────────────────

print(f"{'\u2550' * 50}")
print("  SIGNAL VERIFICATION RESULTS")
print(f"{'\u2550' * 50}")
print(f"  Strategy:   {STRATEGY}")
print(f"  Instrument: {INSTRUMENT_ID}")
print(f"  Interval:   {BAR_INTERVAL}")
print(f"  Periods:    fast={FAST_PERIOD}, slow={SLOW_PERIOD}")
print(f"{'\u2500' * 50}")
print(f"  1. Indicators match:      {'\u2705 PASS' if INDICATORS_MATCH else '\u274c FAIL'}")
print(f"  2. Fills match crossovers: {'\u2705 PASS' if FILLS_MATCH else '\u274c FAIL'}")
print(f"{'\u2500' * 50}")

if INDICATORS_MATCH and FILLS_MATCH:
    print(f"\n  \u2705 ALL CHECKS PASSED \u2014 engine is doing what we expect.")
else:
    print(f"\n  \u274c ISSUES FOUND \u2014 review details above.")
    if not INDICATORS_MATCH:
        print("    \u2022 Indicator values diverge between pandas and NT")
    if not FILLS_MATCH:
        print("    \u2022 Fill timestamps don't align with crossover points")